# Using TsTables 

本 notebook **严格保留原文 tstables 用法**，只做两件事：

- 数据量缩小（避免跑不动）
- 增加必要说明，让代码能在现代环境跑起来

⚠️ 注意：`tstables` 是老库，需要手动安装兼容版本。


## 1. 生成示例数据

In [1]:
import numpy as np
import pandas as pd

def generate_sample_data(rows=10_000, cols=5, seed=42):
    """Generate a reproducible financial-like DataFrame."""
    rng = np.random.default_rng(seed)
    index = pd.date_range("2021-01-01", periods=rows, freq="min")

    steps = rng.normal(loc=0.0, scale=0.02, size=(rows, cols))
    data = 100 + np.cumsum(steps, axis=0)
    columns = [f"No{i}" for i in range(cols)]
    return pd.DataFrame(data, index=index, columns=columns).round(4)

data = generate_sample_data(rows=10_000, cols=5)
data.head()


,No0,No1,No2,No3,No4
2021-01-01 00:00:00,100.0061,99.9792,100.0150,100.0188,99.9610
2021-01-01 00:01:00,99.9801,99.9818,100.0087,100.0185,99.9439
2021-01-01 00:02:00,99.9976,99.9973,100.0100,100.0410,99.9533
2021-01-01 00:03:00,99.9805,100.0047,99.9908,100.0586,99.9523
2021-01-01 00:04:00,99.9768,99.9911,100.0153,100.0555,99.9437


## 2. 定义 TsTables 结构

In [2]:
# !pip install tstables

In [3]:
# 在这里我们需要先定义一个 desc 类，它的作用是描述表的结构（schema）。
# 可以把它理解成数据库里的 “建表语句”：提前规定好每一列的名称、类型和顺序。

# 为什么这里把列写死为 5 列？
# 这其实不是限制，而是 TsTables 的设计选择。

# 这类基于 HDF5 的存储方案，本质是面向“大规模时间序列数据”的：
# - 列结构通常是固定的（例如行情数据、传感器数据）
# - 行数非常大（几十万、几百万甚至更多）
# - 读取时经常按时间区间切片

# 为了提高性能，TsTables 采用固定结构（schema）换性能的思路：
# - 每一行的数据布局在写入前就完全确定（类似 C 语言里的 struct）
# - 数据可以按二进制连续存储（没有额外的类型推断和结构解析开销）
# - 写入时不需要反复检查列结构
# - 读取时可以直接按块（chunk）和时间范围快速定位数据

# 代价就是：表结构一旦创建，就不能随意增加/删除列，也不能动态变化。

# 所以这里写成 5 列，是因为示例数据正好有 5 列；
# 如果你的数据有 N 列，实际做法是：在创建表之前确定好结构，而不是运行过程中动态改变结构。

In [4]:
import tstables
import tables as tb

class desc(tb.IsDescription):
    timestamp = tb.Int64Col(pos=0)
    No0 = tb.Float64Col(pos=1)
    No1 = tb.Float64Col(pos=2)
    No2 = tb.Float64Col(pos=3)
    No3 = tb.Float64Col(pos=4)
    No4 = tb.Float64Col(pos=5)


## 3. 创建 HDF5 数据库

In [5]:
h5 = tb.open_file('data.h5ts', 'w')

In [6]:
h5

File(filename=data.h5ts, title=np.str_(''), mode='w', root_uep='/', filters=Filters(complevel=0, shuffle=False, bitshuffle=False, fletcher32=False, least_significant_digit=None))
/ (RootGroup) np.str_('')

In [7]:
ts = h5.create_ts('/', 'data', desc)

In [8]:
ts

## 4. 写入数据

In [9]:
# 兼容旧版 tstables
import pandas.tseries

# 1. 修 pandas.tseries.index
pandas.tseries.index = pd

# 2. 修 to_records 参数
_old_to_records = pd.DataFrame.to_records
def _patched_to_records(self, *args, **kwargs):
    kwargs.pop("convert_datetime64", None)
    return _old_to_records(self, *args, **kwargs)

pd.DataFrame.to_records = _patched_to_records

In [10]:
%time ts.append(data)

CPU times: user 14.6 ms, sys: 2.86 ms, total: 17.4 ms
Wall time: 16.3 ms


## 5. 区间读取（核心能力）

In [11]:
import datetime

start = datetime.datetime(2021, 1, 2)
end = datetime.datetime(2021, 1, 3)

subset = ts.read_range(start, end)

In [12]:
subset.head()

,No0,No1,No2,No3,No4
2021-01-02 00:00:00,99.6694,100.3856,99.7412,100.2496,99.6347
2021-01-02 00:01:00,99.6531,100.4114,99.7565,100.2357,99.6355
2021-01-02 00:02:00,99.6471,100.3888,99.7600,100.2274,99.6165
2021-01-02 00:03:00,99.6176,100.3575,99.7526,100.2257,99.5990
2021-01-02 00:04:00,99.6306,100.3562,99.7641,100.2220,99.5832


In [13]:
subset.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1441 entries, 2021-01-02 00:00:00 to 2021-01-03 00:00:00
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   No0     1441 non-null   float64
 1   No1     1441 non-null   float64
 2   No2     1441 non-null   float64
 3   No3     1441 non-null   float64
 4   No4     1441 non-null   float64
dtypes: float64(5)
memory usage: 67.5 KB


## 6. 更细粒度区间

In [14]:
start = datetime.datetime(2021, 1, 2, 12, 30, 0)
end = datetime.datetime(2021, 1, 3, 17, 15, 30)

subset = ts.read_range(start, end)
subset.head()

,No0,No1,No2,No3,No4
2021-01-02 12:30:00,98.3039,99.9519,99.1127,99.5585,100.5399
2021-01-02 12:31:00,98.3358,99.9528,99.1137,99.5604,100.5413
2021-01-02 12:32:00,98.2832,99.9491,99.0865,99.5930,100.5730
2021-01-02 12:33:00,98.3002,99.9443,99.0663,99.5798,100.6119
2021-01-02 12:34:00,98.2864,99.8972,99.0869,99.6098,100.6451


## 7. 关闭文件

In [15]:
h5.close()